# Parsing und Exploration der DNB-Hochschulschriften


## 1. Import und Setup

In [ ]:
# Pfad zum src-Ordner hinzufügen
import sys
from pathlib import Path

# Projektroot ermitteln (eine Ebene über notebooks/)
project_root = Path().resolve().parent
src_path = project_root / "src"
sys.path.append(str(src_path))

print("src-Pfad hinzugefügt:", src_path)


In [ ]:
# Eigene Module
from core.marc21_parser import parse_dnb_theses
from core.data_explorer import Marc21Explorer

# Standardbibliotheken
import pandas as pd
from collections import Counter
from pathlib import Path

# Datenanalyse
from IPython.display import display

# Visualisierung
import matplotlib.pyplot as plt
import seaborn as sns  # optional, für schönere Plots

# Anzeigeoptionen
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
sns.set_theme(style="whitegrid")

## 2. Daten einlesen und speichern

### 2.1 Auswahl der Datensätze
NB: Am 12.02.2026 Total records parsed: 1619129 in 10min+, am 16.02. in 8m 20.6s

In [ ]:
# --- Relativer Pfad zur Datei im Repository ---
data_file = Path("../data/raw/dnb-all_hochschulschriften_dnbmarc.mrc.xml")

# --- Prüfen, ob die Datei existiert ---
if not data_file.exists():
    print("Die MARC21-Datei wurde nicht gefunden!")
    print("Bitte lade die Datei von DNB Lab herunter:")
    print("https://www.dnb.de/dnblabdatendiss")
    #print(f"Speichere sie hier: {data_file.resolve()}")
else:
    # --- Parser aufrufen ---
    df = parse_dnb_theses(
        filepath=str(data_file),
        limit=100000,   # optional, z.B. zum Testen
        verbose=True
    )
    print("Datei geladen!") 


In [ ]:
  # --- Erste Übersicht ---
display(df.head().T)
print(df.info())

### 2.2 Speichern

In [ ]:
# --- Parquet-Pfad ---
parquet_file = Path("../data/raw/dnb_theses.parquet")

# --- DataFrame speichern ---
df.to_parquet(parquet_file, engine="pyarrow", compression="snappy")
print(f"DataFrame gespeichert als: {parquet_file.resolve()}")

# --- Für zukünftige Notebooks: nur Parquet laden ---
# df = pd.read_parquet(parquet_file)


In [ ]:
#import sys
#print(sys.executable)

#import pyarrow
#print(pyarrow.__version__)

In [ ]:
# --- Optional: Als CSV speichern ---
# csv_file = Path("../data/raw/dnb_theses.csv")
##csv_file.parent.mkdir(parents=True, exist_ok=True)
# df.to_csv(csv_file, index=False, encoding='utf-8')
# print(f"DataFrame als CSV gespeichert: {csv_file.resolve()}")

## 3. Datenanalyse

In [ ]:
# --- Für zukünftige Notebooks: nur Parquet laden ---
# df = pd.read_parquet(parquet_file)

In [ ]:
# Explorer initialisieren
explorer = Marc21Explorer(df)

In [ ]:
# Test, ob Listen in DataFrame-Zellen korrekt gespeichert wurden
print("Datentyp der 084_list-Zelle:", type(explorer.df["084_list"].iloc[3]))

# wenn String, dann mit json.loads() zurück in Liste umwandeln
# dann in der Explorer-Klasse anpassen, damit das automatisch passiert

In [ ]:
# Übersicht über die Spalten 
display(explorer.overview(max_elements_preview=3))

# Fehlende Werte
missing = explorer.missing_report()
display(missing)

### 3.1 Klassifizierung

In [ ]:
# Relevante MARC-Felder
marc_list_fields = ["084_list", "083_list", "082_list"]

total_records = len(df)

for field in marc_list_fields:
    if field in df.columns:
        # Zählen, wie viele Zellen nicht leer sind (Liste enthält mindestens ein Element)
        count_non_empty = df[field].dropna().apply(lambda x: len(x) > 0).sum()
        percent = (count_non_empty / total_records) * 100
        print(f"{field}: {count_non_empty}/{total_records} Datensätze ({percent:.1f}%) mit Einträgen")

# 084_list: 747468/1619129 Datensätze (46.2%) mit Einträgen
# 083_list: 549580/1619129 Datensätze (33.9%) mit Einträgen
# 082_list: 812149/1619129 Datensätze (50.2%) mit Einträgen

In [ ]:
# Liste für die aufgeteilten Daten
data = []

# Durch alle Datensätze gehen
for recs in df['084_list'].dropna():  # alle nicht-leeren Zellen
    if isinstance(recs, list):
        for subfield_dict in recs:
            a_value = subfield_dict.get('a')
            if a_value:
                # Aufteilen, falls mehrere Codes in einem String (getrennt durch ';')
                codes = [code.strip() for code in a_value.split(';')]
                for code in codes:
                    data.append({
                        'a': code,
                        'q': subfield_dict.get('q'),
                        '2': subfield_dict.get('2')
                    })

# In DataFrame umwandeln
df_084 = pd.DataFrame(data)

# Überblick
print("Beispielhafte Vorschau auf die Daten:")
print(df_084.head())

# Anzahl der Datensätze nach Vergabestelle oder Quelle zählen
print("\nAnzahl der Notationen pro Vergabestelle (q):")
print(df_084['q'].value_counts())

print("\nAnzahl der Notationen pro Quelle (2):")
print(df_084['2'].value_counts())


In [ ]:
# Gesamtanzahl aller Notationen
total_notations = len(df_084)

# Top 10 Notationen
top_10 = df_084['a'].value_counts().head(10)

# Ausgabe mit Anzahl und prozentualem Anteil
print("Top 10 Notationen in 084_list $a:")
for code, count in top_10.items():
    percent = (count / total_notations) * 100
    print(f"{code}: {count} ({percent:.1f}%)")

 # Achtung irreführend, weil verschiedene Notationssysteme (Dewey, RVK, LCC) in einem Feld vermischt sein können.   


In [ ]:
# Dataframe mit aufgeteilten Feldern vorbereiten
marc_list_fields = ["082_list", "083_list", "084_list"]

# Hilfsfunktion, um $a, $q, $2 aus einem list[dict] zu extrahieren
def extract_subfields(sublist, subfields=['a','q','2']):
    if sublist is None or not isinstance(sublist, list) or len(sublist) == 0:
        return {f: None for f in subfields}
    
    result = {f: [] for f in subfields}
    for entry in sublist:
        for f in subfields:
            if f in entry and entry[f]:
                # Trennen, falls mehrere Codes in $a z. B. "1701 ; 900"
                if f == 'a' and ';' in entry[f]:
                    codes = [c.strip() for c in entry[f].split(';')]
                    result[f].extend(codes)
                else:
                    result[f].append(entry[f])
    # Falls leere Liste → None
    for f in subfields:
        if not result[f]:
            result[f] = None
    return result

# Neues DataFrame vorbereiten
data = []

for idx, row in df.iterrows():
    new_row = {'record_id': row['record_id']}
    
    # 082
    if '082_list' in df.columns:
        extracted = extract_subfields(row.get('082_list'))
        new_row['082_a'] = extracted['a']
        new_row['082_q'] = extracted['q']
        new_row['082_2'] = extracted['2']
    
    # 083
    if '083_list' in df.columns:
        extracted = extract_subfields(row.get('083_list'))
        new_row['083_a'] = extracted['a']
        new_row['083_q'] = extracted['q']
        new_row['083_2'] = extracted['2']
    
    # 084
    if '084_list' in df.columns:
        extracted = extract_subfields(row.get('084_list'))
        new_row['084_a'] = extracted['a']
        new_row['084_q'] = extracted['q']
        new_row['084_2'] = extracted['2']
    
    data.append(new_row)

df_flat = pd.DataFrame(data)



In [ ]:
df_flat


In [ ]:

# Beispiel-Notation-Dictionary (kann beliebig erweitert werden)
notation_dict = {
    "1701": "Allgemeine Geschichte",
    "900": "Rechtswissenschaften",
    "920": "Philosophie",
    "940": "Theologie",
    "943": "Geschichte Deutschlands",
    "950": "Naturwissenschaften",
    "960": "Technik",
    "970": "Kunst",
    "980": "Musik",
    "990": "Literatur",
    "2000": "Technik / Ingenieurwesen",
    "320": "Politik / Staatswissenschaften",
    "340": "Recht / Gesetzgebung",
    "330": "Soziologie / Sozialwissenschaften",
    "0300": "Geographie / Kartographie",
    "0400": "Bibliothekswesen / Dokumentation",
    "510": "Mathematik",
    "530": "Physik",
    "540": "Chemie",
    "610": "Bauwesen / Architektur",
    "620": "Maschinenbau",
    "33": "Politik / Staatswissenschaften",
    "1501": "Geschichte Deutschlands",
    # ... erweitern je nach Bedarf
}

# 1️⃣ SDNB-Datensätze filtern
df_sdnb = df_flat[df_flat['084_2'].apply(lambda x: isinstance(x, list) and 'sdnb' in x)]

# 2️⃣ Alle 084_a flatten
all_codes = []
for codes in df_sdnb['084_a'].dropna():
    all_codes.extend(codes)

# 3️⃣ Häufigkeiten zählen
counter = Counter(all_codes)
total_codes = sum(counter.values())
top_10 = counter.most_common(10)

# 4️⃣ Ausgabe mit Bedeutungen und Prozent
print("Top 10 SDNB-Notationen in 084_list $a mit Bedeutungen und Prozent:")
for code, count in top_10:
    meaning = notation_dict.get(code, "Unbekannt")
    percent = (count / total_codes) * 100
    print(f"{code}: {count} ({percent:.1f}%) – {meaning}")

# 5️⃣ Visualisierung Top-10
top_codes = [c for c, _ in top_10]
top_counts = [c for _, c in top_10]
top_meanings = [notation_dict.get(c, "Unbekannt") for c in top_codes]

plt.figure(figsize=(12,6))
sns.barplot(x=top_counts, y=top_meanings, palette="viridis")
plt.xlabel("Anzahl Notationen")
plt.ylabel("Bedeutung der Notation")
plt.title("Top 10 SDNB-Notationen in 084_list $a")
plt.show()

# 6️⃣ Vergleich der Anzahl Datensätze pro Quelle in 084_2
# flatten alle Quellen
all_sources = []
for sources in df_flat['084_2'].dropna():
    all_sources.extend(sources)

source_counts = Counter(all_sources)
print("\nAnzahl Datensätze pro Quelle in 084_list:")
for src, count in source_counts.items():
    print(f"{src}: {count}")

# Optional: Visualisierung Quellen
plt.figure(figsize=(8,4))
sns.barplot(x=list(source_counts.keys()), y=list(source_counts.values()), palette="magma")
plt.ylabel("Anzahl Datensätze")
plt.xlabel("Quelle")
plt.title("Datensätze pro Quelle in 084_list")
plt.show()


### 3.2 Dissertationsvermerk

In [ ]:
def count_dissertation_notes(df: pd.DataFrame) -> int:
    """
    Zählt die Datensätze in 'dissertation_note', die nicht leer sind.
    """
    return df["dissertation_note"].apply(lambda x: bool(str(x).strip())).sum()


In [ ]:
def dissertation_note_overview(df: pd.DataFrame) -> pd.DataFrame:
    """
    Übersicht über vorhandene / fehlende Dissertationseinträge.
    """
    total = len(df)

    def has_note(x):
        if pd.isna(x):
            return False
        if isinstance(x, str) and x.strip() == "":
            return False
        return True

    num_with_entry = df["dissertation_note"].apply(has_note).sum()
    num_empty = total - num_with_entry

    return pd.DataFrame({
        "category": ["Mit Dissertationseintrag", "Ohne Dissertationseintrag"],
        "count": [num_with_entry, num_empty],
        "percent": [round(num_with_entry/total*100,1), round(num_empty/total*100,1)]
    })


In [ ]:
overview = dissertation_note_overview(df)
print(overview)


In [ ]:
# Übersicht erstellen
overview = dissertation_note_overview(df)
print(overview)

# Plot
categories = overview["category"]
counts = overview["count"]

plt.bar(categories, counts, color=['steelblue', 'salmon'])
plt.ylabel('Anzahl')
plt.title('Übersicht Dissertationseintrag')
plt.show()


In [ ]:
# Extraktion der Promotionsjahre
df["promotionsjahr"] = df["dissertation_note"].str.extract(r"(\d{4})")

# Überprüfen, ob die Extraktion funktioniert hat
print(df[["dissertation_note"]].head(10))

In [ ]:
# Zählen, wie viele Datensätze 'Fak.' enthalten
anzahl_fak = df["dissertation_note"].str.contains("Fak", case=False, na=False).sum()
gesamt = len(df)

print(f"Anzahl der Datensätze mit 'Fak': {anzahl_fak} von insgesamt {gesamt} Titeln")

### 3.3 Zeitliche Entwicklung

In [ ]:
# 1. In Zahlen umwandeln, fehlerhafte Werte -> NaN
df["publication_year"] = pd.to_numeric(df["publication_year"], errors="coerce")

# 2. Nur plausible Jahre behalten (NaN wird automatisch entfernt)
df_filtered = df["publication_year"][(df["publication_year"] >= 1900) & (df["publication_year"] <= 2026)]

# 3. Jahr-Häufigkeiten zählen
year_counts = df_filtered.value_counts().sort_index()

# 4. Jahre in Jahrzehnte umwandeln und summieren
decade_counts = year_counts.groupby((year_counts.index // 10) * 10).sum()

# 5. Nur Jahrzehnte mit mindestens einer Publikation behalten
decade_counts = decade_counts[decade_counts > 0]

# 6. Plot
plt.figure(figsize=(12, 6))
plt.bar(
    x=[f"{int(d)}er" for d in decade_counts.index],  # schöne Labels
    height=decade_counts.values,
    color='skyblue'
)
plt.title('Anzahl der Veröffentlichungen nach Jahrzehnt')
plt.xlabel('Jahrzehnt')
plt.ylabel('Anzahl')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Jahr prüfen
def check_year(year, df):
    """
    Prüft, ob ein Jahr in df['publication_year'] enthalten ist.
    Gibt die Zeilen mit diesem Jahr zurück.
    """
    if year in df["publication_year"].values:
        print(f"{year} ist in df['publication_year'] enthalten")
        # Zeilen mit diesem Jahr ausgeben
        print(df[df["publication_year"] == year])
    else:
        print(f"{year} ist nicht in df['publication_year'] enthalten")

# Beispiele:
check_year(1945, df)


### 3.4 Schlagwörter/Themen

In [ ]:
# Liste der Spalten, die Schlagwörter enthalten
keyword_columns = ['648_list', '650_list', '830_list']

# Anzahl Schlagwörter pro Zeile berechnen
df['subject_count'] = df[keyword_columns].apply(lambda row: sum(bool(x) for x in row), axis=1)

# Statistik ausgeben
subject_stats = df['subject_count']
print(f"Durchschnittliche Anzahl Schlagwörter: {subject_stats.mean():.2f}")
print(f"Median der Schlagwörter: {subject_stats.median():.0f}")
print(f"Maximale Anzahl Schlagwörter: {subject_stats.max():.0f}")
print(f"Minimale Anzahl Schlagwörter: {subject_stats.min():.0f}")
   

## 4. Daten filtern

In [ ]:
# Sicherstellen, dass publication_year numerisch ist
df['publication_year'] = pd.to_numeric(df['publication_year'], errors='coerce')

# --- Einzelnes Jahr ---
jahr = 1939
df_filtered = df[df['publication_year'] == jahr]
print(f"Dissertationen aus dem Jahr {jahr}: {len(df_filtered)}")

# --- Liste von Jahren ---
jahre_liste = [1955, 1956, 1958]
df_filtered = df[df['publication_year'].isin(jahre_liste)]
print(f"Dissertationen aus den Jahren {jahre_liste}: {len(df_filtered)}")

# --- Jahrbereich ---
start_jahr = 1913
end_jahr = 1939
df_filtered = df[(df['publication_year'] >= start_jahr) & (df['publication_year'] <= end_jahr)]
print(f"Dissertationen von {start_jahr} bis {end_jahr}: {len(df_filtered)}")


In [ ]:
# Beispiel: Dissertationen aus Leipzig
df_leipzig = df[df['publication_place'].str.contains('Leipzig', na=False, case=False)]
print(f"Dissertationen aus Leipzig: {len(df_leipzig)}")
df_leipzig[['author_name', 'title', 'publication_year']].head(10)

## 5. Erweiterte Analyse: Titel

In [ ]:
# Titel-Längen analysieren
df['title_length'] = df['title'].str.len()
print(f"Durchschnittliche Titel-Länge: {df['title_length'].mean():.0f} Zeichen")
print(f"Längster Titel: {df['title_length'].max():.0f} Zeichen")
print(f"Kürzester Titel: {df['title_length'].min():.0f} Zeichen")

In [ ]:
# Längster Titel anzeigen
# longest_title = df.loc[df['title_length'].idxmax()]
# print("Längster Titel:")
# print(f"Autor: {longest_title['author_name']}")
# print(f"Titel: {longest_title['title']}")
# print(f"Jahr: {longest_title['publication_year']}")

In [ ]:
# Suchbegriff: Mineralog*
search_term = "Mineralog"

# Filtere nach 'title' oder 'title_remainder'
filtered_df = df[
    df['title'].str.contains(search_term, case=False, regex=True) |
    df['title_remainder'].str.contains(search_term, case=False, regex=True)
]

# Optional: nur bestimmte Spalten anzeigen
result_df = filtered_df[['author_name', 'title', 'title_remainder', 'publication_year']].copy()

# Anzahl der Treffer ausgeben
num_results = len(result_df)
print(f"Anzahl gefundener Titel mit '{search_term}': {num_results}")

# DataFrame ausgeben
result_df


In [ ]:
df['publication_year'].notna().sum()


## Playground

In [ ]:
marc_list_fields = ["082_list", "083_list", "084_list"]

def normalize_marc_lists(df, fields=marc_list_fields):
    rows = []
    
    for _, row in df.iterrows():
        record_id = row['record_id']
        
        for field in fields:
            sublist = row.get(field)
            if not sublist:
                continue
            for entry in sublist:
                # $a aufteilen, falls mehrere Werte durch ';' getrennt
                a_values = [v.strip() for v in entry.get('a','').split(';')] if entry.get('a') else [None]
                
                for a_val in a_values:
                    rows.append({
                        'record_id': record_id,
                        'marc_field': field[:3],  # z.B. '082'
                        'a': a_val,
                        'q': entry.get('q'),
                        '2': entry.get('2')
                    })
    
    return pd.DataFrame(rows)

# Normalisierte Tabelle erstellen
df_normalized = normalize_marc_lists(df)

# Als CSV speichern
df_normalized.to_csv("marc_normalized.csv", index=False, encoding='utf-8-sig')
print("Datei 'marc_normalized.csv' erfolgreich gespeichert!")


In [ ]:
# Normalisiertes MARC DataFrame df_normalized
# Spalten: record_id, marc_field, a, q, 2

# 1. Nur eindeutige $a-Codes extrahieren
lookup_table = df_normalized[['a', 'q', '2']].drop_duplicates().reset_index(drop=True)

# Optional: Spalte mit Kategorie oder Typ erstellen (z.B. marc_field)
lookup_table['category'] = df_normalized.groupby('a')['marc_field'].first().reindex(lookup_table['a']).values

# Ergebnis als CSV speichern
lookup_table.to_csv("a_lookup_table.csv", index=False, encoding='utf-8-sig')
print("Lookup-Tabelle erstellt und gespeichert!")


In [ ]:
# Join auf $a
df_with_features = df_normalized.merge(lookup_table, on='a', how='left')


In [ ]:
# Parquet speichern
# df_with_features.to_parquet("df_with_features.parquet", index=False)

df_with_features
